In [1]:
#Import necessary packages 

In [2]:
import os
import json
import numpy
import datetime
import certifi
import pandas as pd

import pymongo
import sqlalchemy
from sqlalchemy import create_engine, text

In [3]:
import findspark
findspark.init()
print(findspark.find())

from pyspark import SparkConf
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window as W

/opt/anaconda3/lib/python3.13/site-packages/pyspark


In [4]:
#Verify whether Pymongo and SQL Alchemy are up to date

In [5]:
print(f"Running SQL Alchemy Version: {sqlalchemy.__version__}")
print(f"Running PyMongo Version: {pymongo.__version__}")

Running SQL Alchemy Version: 2.0.43
Running PyMongo Version: 4.16.0


In [6]:
#Define input variables for sql and mongo connections

In [7]:
# --------------------------------------------------------------------------------
# Specify MySQL Server Connection Information
# --------------------------------------------------------------------------------
mysql_args = {
    "host_name" : "localhost",
    "port" : "3306",
    "db_name" : "adventureworks_dw",
    "conn_props" : {
        "user" : "root",
        "password" : "Password",
        "driver" : "com.mysql.cj.jdbc.Driver"
    }
}

# --------------------------------------------------------------------------------
# Specify MongoDB Cluster Connection Information
# --------------------------------------------------------------------------------
mongodb_args = {
    "cluster_location" : "atlas", # "atlas"
    "user_name" : "anweshachowdhury243_db_user",
    "password" : "1GjR5oL4ahO3htrw",
    "cluster_name" : "cluster0",
    "cluster_subnet" : "ks9qt5b",
    "db_name" : "adventureworks_dw",
    "collection" : "",
    "null_column_threshold" : 0.5
}
mysql_args2 = {
    "uid" : "root",
    "pwd" : "Password",
    "hostname" : "localhost",  #"wna8fw-mysql.mysql.database.azure.com",
    "dbname" : "adventureworks_dw"
}

# The 'cluster_location' must either be "atlas" or "local".
mongodb_args2 = {
    "user_name" : "anweshachowdhury243_db_user", #db_user
    "password" : "1GjR5oL4ahO3htrw", #db_password
    "cluster_name" : "cluster0",
    "cluster_subnet" : "ks9qt5b",
    "cluster_location" : "atlas", # "local"
    "db_name" : "adventureworks"
}
# --------------------------------------------------------------------------------
# Specify Directory Structure for Source Data
# --------------------------------------------------------------------------------
data_dir = os.path.join(os.getcwd(), 'data')
batch_dir = os.path.join(data_dir, 'batch')
stream_dir = os.path.join(data_dir, 'streaming')

sales_orders_stream_dir = os.path.join(stream_dir, 'sales_orders')
purchase_orders_stream_dir = os.path.join(stream_dir, 'purchase_orders')

# --------------------------------------------------------------------------------
# Create Directory Structure for Data Lakehouse Files
# --------------------------------------------------------------------------------

dest_database = "adventureworks_dlh"
sql_warehouse_dir = os.path.abspath('spark-warehouse')
dest_database_dir = f"{dest_database}.db"
database_dir = os.path.join(sql_warehouse_dir, dest_database_dir)

# fact_purchase_orders
purchase_orders_output_bronze = os.path.join(database_dir, 'fact_purchase_orders', 'bronze')
purchase_orders_output_silver = os.path.join(database_dir, 'fact_purchase_orders', 'silver')
purchase_orders_output_gold   = os.path.join(database_dir, 'fact_purchase_orders', 'gold')

In [8]:
#Assign source and destination database

In [9]:
host_name = "localhost"
port = "3306"
user_id = "root"
pwd = "Password"

src_dbname = "adventureworks"
dst_dbname = "adventureworks_dw"

In [10]:
#Define functions

In [11]:
def get_file_info(path: str):
    file_sizes = []
    modification_times = []

    '''Fetch each item in the directory, and filter out any directories.'''
    items = os.listdir(path)
    files = sorted([item for item in items if os.path.isfile(os.path.join(path, item))])

    '''Populate lists with the Size and Last Modification DateTime for each file in the directory.'''
    for file in files:
        file_sizes.append(os.path.getsize(os.path.join(path, file)))
        modification_times.append(pd.to_datetime(os.path.getmtime(os.path.join(path, file)), unit='s'))

    data = list(zip(files, file_sizes, modification_times))
    column_names = ['name','size','modification_time']
    
    return pd.DataFrame(data=data, columns=column_names)


def wait_until_stream_is_ready(query, min_batches=1):
    while len(query.recentProgress) < min_batches:
        time.sleep(5)
        
    print(f"The stream has processed {len(query.recentProgress)} batchs")


def remove_directory_tree(path: str):
    '''If it exists, remove the entire contents of a directory structure at a given 'path' parameter's location.'''
    try:
        if os.path.exists(path):
            shutil.rmtree(path)
            return f"Directory '{path}' has been removed successfully."
        else:
            return f"Directory '{path}' does not exist."
            
    except Exception as e:
        return f"An error occurred: {e}"
        

def drop_null_columns(df, threshold):
    '''Drop Columns having a percentage of NULL values that exceeds the given 'threshold' parameter value.'''
    columns_with_nulls = [col for col in df.columns if df.filter(df[col].isNull()).count() / df.count() > threshold] 
    df_dropped = df.drop(*columns_with_nulls) 
    
    return df_dropped
    
    
def get_mysql_dataframe(spark_session, sql_query : str, **args):
    '''Create a JDBC URL to the MySQL Database'''
    jdbc_url = f"jdbc:mysql://{args['host_name']}:{args['port']}/{args['db_name']}"
    
    '''Invoke the spark.read.format("jdbc") function to query the database, and fill a DataFrame.'''
    dframe = spark_session.read.format("jdbc") \
    .option("url", jdbc_url) \
    .option("driver", args['conn_props']['driver']) \
    .option("user", args['conn_props']['user']) \
    .option("password", args['conn_props']['password']) \
    .option("query", sql_query) \
    .load()
    
    return dframe
    

def get_mongo_uri(**args):
    '''Validate proper input'''
    if args["cluster_location"] not in ['atlas', 'local']:
        raise Exception("You must specify either 'atlas' or 'local' for the 'cluster_location' parameter.")
        
    if args['cluster_location'] == "atlas":
        uri = f"mongodb+srv://{args['user_name']}:{args['password']}@"
        uri += f"{args['cluster_name']}.{args['cluster_subnet']}.mongodb.net/"
    else:
        uri = "mongodb://localhost:27017/"

    return uri


def get_spark_conf_args(spark_jars : list, **args):
    jars = ""
    for jar in spark_jars:
        jars += f"{jar}, "
    
    sparkConf_args = {
        "app_name" : "PySpark Northwind Data Lakehouse (Medallion Architecture)",
        "worker_threads" : f"local[{int(os.cpu_count()/2)}]",
        "shuffle_partitions" : int(os.cpu_count()),
        "mongo_uri" : get_mongo_uri(**args),
        "spark_jars" : jars[0:-2],
        "database_dir" : sql_warehouse_dir
    }
    
    return sparkConf_args
    

def get_spark_conf(**args):
    sparkConf = SparkConf().setAppName(args['app_name'])\
    .setMaster(args['worker_threads']) \
    .set('spark.driver.memory', '4g') \
    .set('spark.executor.memory', '2g') \
    .set('spark.jars', args['spark_jars']) \
    .set('spark.jars.packages', 'org.mongodb.spark:mongo-spark-connector_2.12:3.0.1') \
    .set('spark.mongodb.input.uri', args['mongo_uri']) \
    .set('spark.mongodb.output.uri', args['mongo_uri']) \
    .set('spark.sql.adaptive.enabled', 'false') \
    .set('spark.sql.debug.maxToStringFields', 35) \
    .set('spark.sql.shuffle.partitions', args['shuffle_partitions']) \
    .set('spark.sql.streaming.forceDeleteTempCheckpointLocation', 'true') \
    .set('spark.sql.streaming.schemaInference', 'true') \
    .set('spark.sql.warehouse.dir', args['database_dir']) \
    .set('spark.streaming.stopGracefullyOnShutdown', 'true')
    
    return sparkConf


def get_mongo_client(**args):
    '''Get MongoDB Client Connection'''
    mongo_uri = get_mongo_uri(**args)
    if args['cluster_location'] == "atlas":
        client = pymongo.MongoClient(mongo_uri, tlsCAFile=certifi.where())

    elif args['cluster_location'] == "local":
        client = pymongo.MongoClient(mongo_uri)
        
    else:
        raise Exception("A MongoDB Client could not be created.")

    return client
    
    
# TODO: Rewrite this to leverage PySpark?
def set_mongo_collections(mongo_client, db_name : str, data_directory : str, json_files : list):
    db = mongo_client[db_name]
    
    for file in json_files:
        db.drop_collection(file)
        json_file = os.path.join(data_directory, json_files[file])
        with open(json_file, 'r') as openfile:
            json_object = json.load(openfile)
            file = db[file]
            result = file.insert_many(json_object)
        
    mongo_client.close()
    

def get_mongodb_dataframe(spark_session, **args):
    '''Query MongoDB, and create a DataFrame'''
    dframe = spark_session.read.format("com.mongodb.spark.sql.DefaultSource") \
        .option("database", args['db_name']) \
        .option("collection", args['collection']).load()

    '''Drop the '_id' index column to clean up the response.'''
    dframe = dframe.drop('_id')
    
    '''Call the drop_null_columns() function passing in the dataframe.'''
    dframe = drop_null_columns(dframe, args['null_column_threshold'])
    
    return dframe

In [12]:
def get_sql_dataframe(sql_query, **args):
    '''Create a connection to the MySQL database'''
    conn_str = f"mysql+pymysql://{args['uid']}:{args['pwd']}@{args['hostname']}/{args['dbname']}"
    sqlEngine = create_engine(conn_str, pool_recycle=3600)
    connection = sqlEngine.connect()
    
    '''Invoke the pd.read_sql() function to query the database, and fill a Pandas DataFrame.'''
    dframe = pd.read_sql(text(sql_query), connection);
    connection.close()
    
    return dframe
    

def set_dataframe(user_id, pwd, host_name, db_name, df, table_name, pk_column, db_operation):
    conn_str = f"mysql+pymysql://{user_id}:{pwd}@{host_name}/{db_name}"
    sqlEngine = create_engine(conn_str, pool_recycle=3600)
    db_connection = sqlEngine.connect()
    
    '''Invoke the Pandas DataFrame .to_sql( ) function to either create, or append to, a table'''
    if db_operation in ['insert', 'update']:
        if db_operation.lower() == "insert":
            df.to_sql(table_name, con=db_connection, index=False, if_exists='replace')
            db_connection.execute(text(f"ALTER TABLE {table_name} ADD {pk_column} INT AUTO_INCREMENT PRIMARY KEY FIRST;"))
                    
        elif db_operation.lower() == "update":
            df.to_sql(table_name, con=db_connection, index=False, if_exists='append')

    else:
        print("The value supplied to the 'db_operation' parameter must be either 'insert' or 'update'.")
    
    db_connection.close()


def get_mongo_client(**args):
    '''Validate proper input'''
    if args["cluster_location"] not in ['atlas', 'local']:
        raise Exception("You must specify either 'atlas' or 'local' for the cluster_location parameter.")
    
    else:
        if args["cluster_location"] == "atlas":
            connect_str = f"mongodb+srv://{args['user_name']}:{args['password']}@"
            connect_str += f"{args['cluster_name']}.{args['cluster_subnet']}.mongodb.net"
            client = pymongo.MongoClient(connect_str, tlsCAFile=certifi.where())
            
        elif args["cluster_location"] == "local":
            client = pymongo.MongoClient("mongodb://localhost:27017/")
        
    return client


def get_mongo_dataframe2(mongo_client, db_name, collection, query):
    '''Query MongoDB, and fill a python list with documents to create a DataFrame'''
    db = mongo_client[db_name]
    dframe = pd.DataFrame(list(db[collection].find(query)))
    dframe.drop(['_id'], axis=1, inplace=True)
    mongo_client.close()
    
    return dframe


def set_mongo_collections2(mongo_client, db_name, data_directory, json_files):
    db = mongo_client[db_name]
    
    for file in json_files:
        db.drop_collection(file)
        json_file = os.path.join(data_directory, json_files[file])
        with open(json_file, 'r') as openfile:
            json_object = json.load(openfile)
            file = db[file]
            result = file.insert_many(json_object)
        
    mongo_client.close()

In [13]:
#Create destination database: adventureworks_dw

In [14]:
conn_str = f"mysql+pymysql://{user_id}:{pwd}@{host_name}"
sqlEngine = create_engine(conn_str, pool_recycle=3600)
connection = sqlEngine.connect()

connection.execute(text(f"DROP DATABASE IF EXISTS `{dst_dbname}`;"))
connection.execute(text(f"CREATE DATABASE `{dst_dbname}`;"))
connection.execute(text(f"USE {dst_dbname};"))

connection.close()

In [26]:
remove_directory_tree(database_dir)

"An error occurred: name 'shutil' is not defined"

In [16]:
worker_threads = f"local[{int(os.cpu_count()/2)}]"

jars = []
mysql_spark_jar = os.path.join(os.getcwd(), "mysql-connector-j-9.1.0", "mysql-connector-j-9.1.0.jar")
mssql_spark_jar = os.path.join(os.getcwd(), "sqljdbc_12.8", "enu", "jars", "mssql-jdbc-12.8.1.jre11.jar")

jars.append(mysql_spark_jar)
#jars.append(mssql_spark_jar)

sparkConf_args = get_spark_conf_args(jars, **mongodb_args)

sparkConf = get_spark_conf(**sparkConf_args)
spark = SparkSession.builder.config(conf=sparkConf).getOrCreate()
spark.sparkContext.setLogLevel("OFF")
spark

26/05/06 22:52:51 WARN Utils: Your hostname, Trinas-MacBook-Air.local resolves to a loopback address: 127.0.0.1; using 172.25.62.11 instead (on interface en0)
26/05/06 22:52:51 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Ivy Default Cache set to: /Users/trinachowdhury/.ivy2/cache
The jars for the packages stored in: /Users/trinachowdhury/.ivy2/jars
org.mongodb.spark#mongo-spark-connector_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-eaf0cb54-a95d-42c5-9ae6-7ac45be606e6;1.0
	confs: [default]
	found org.mongodb.spark#mongo-spark-connector_2.12;3.0.1 in central
	found org.mongodb#mongodb-driver-sync;4.0.5 in central
	found org.mongodb#bson;4.0.5 in central
	found org.mongodb#mongodb-driver-core;4.0.5 in central


:: loading settings :: url = jar:file:/opt/anaconda3/lib/python3.13/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


:: resolution report :: resolve 87ms :: artifacts dl 2ms
	:: modules in use:
	org.mongodb#bson;4.0.5 from central in [default]
	org.mongodb#mongodb-driver-core;4.0.5 from central in [default]
	org.mongodb#mongodb-driver-sync;4.0.5 from central in [default]
	org.mongodb.spark#mongo-spark-connector_2.12;3.0.1 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   4   |   0   |   0   |   0   ||   4   |   0   |
	---------------------------------------------------------------------
:: retrieving :: org.apache.spark#spark-submit-parent-eaf0cb54-a95d-42c5-9ae6-7ac45be606e6
	confs: [default]
	0 artifacts copied, 4 already retrieved (0kB/2ms)
26/05/06 22:52:52 WARN NativeCodeLoader: Unable to load native-hadoop library f

In [27]:
spark.sql(f"DROP DATABASE IF EXISTS {dest_database} CASCADE;")

sql_create_db = f"""
    CREATE DATABASE IF NOT EXISTS {dest_database}
    COMMENT 'DS-2002 Lab 06 Database'
    WITH DBPROPERTIES (contains_pii = true, purpose = 'DS-2002 Lab 6.0');
"""
spark.sql(sql_create_db)

DataFrame[]

In [28]:
get_file_info(batch_dir)

,name,size,modification_time
0,.DS_Store,6148,2026-05-06 18:40:28.075867414
1,dim_products_vw.json,327601,2026-03-23 16:02:59.438748121
2,dim_products_vw.json.csv,185555,2026-03-23 16:30:02.877384186
3,fact_purchase_orders_vw.csv,185555,2026-03-23 16:31:34.577282190
4,trips.json,7112796,2026-01-14 02:21:28.476708651


In [29]:
#Populate products dimension from NoSQL database (MongoDB) - Dimension 1 

In [30]:
client = get_mongo_client(**mongodb_args)

json_files = {"products" : 'dim_products_vw.json'}

set_mongo_collections(client, mongodb_args["db_name"], batch_dir, json_files)   

In [21]:
mongodb_args["collection"] = "products"

df_products = get_mongodb_dataframe(spark, **mongodb_args)
df_products.toPandas().head(2)

,Color,DaysToManufacture,FinishedGoodsFlag,ListPrice,MakeFlag,Name,ProductCategory,ProductID,ProductLine,ProductModel,ProductNumber,ProductSubcategory,ReorderPoint,SafetyStockLevel,SellStartDate,StandardCost
0,None,0,0,0.0,0,Adjustable Race,None,1,None,None,AR-5381,None,750,1000,1998-06-01 00:00:00,0.0
1,None,0,0,0.0,0,Bearing Ball,None,2,None,None,BA-8327,None,750,1000,1998-06-01 00:00:00,0.0


In [22]:
drop_cols = ['Class','Style','ProductSubcategory']
df_products.drop(*drop_cols)

DataFrame[Color: string, DaysToManufacture: int, FinishedGoodsFlag: string, ListPrice: double, MakeFlag: string, Name: string, ProductCategory: string, ProductID: int, ProductLine: string, ProductModel: string, ProductNumber: string, ReorderPoint: int, SafetyStockLevel: int, SellStartDate: string, StandardCost: double]

In [31]:
df_products.write.saveAsTable(f"{dest_database}.products", mode="overwrite")

In [32]:
spark.sql(f"DESCRIBE EXTENDED {dest_database}.products;").show()
spark.sql(f"SELECT * FROM {dest_database}.products LIMIT 2").toPandas()

+--------------------+------------------+-------+
|            col_name|         data_type|comment|
+--------------------+------------------+-------+
|               Color|            string|   NULL|
|   DaysToManufacture|               int|   NULL|
|   FinishedGoodsFlag|            string|   NULL|
|           ListPrice|            double|   NULL|
|            MakeFlag|            string|   NULL|
|                Name|            string|   NULL|
|     ProductCategory|            string|   NULL|
|           ProductID|               int|   NULL|
|         ProductLine|            string|   NULL|
|        ProductModel|            string|   NULL|
|       ProductNumber|            string|   NULL|
|  ProductSubcategory|            string|   NULL|
|        ReorderPoint|               int|   NULL|
|    SafetyStockLevel|               int|   NULL|
|       SellStartDate|            string|   NULL|
|        StandardCost|            double|   NULL|
|                    |                  |       |


,Color,DaysToManufacture,FinishedGoodsFlag,ListPrice,MakeFlag,Name,ProductCategory,ProductID,ProductLine,ProductModel,ProductNumber,ProductSubcategory,ReorderPoint,SafetyStockLevel,SellStartDate,StandardCost
0,None,0,0,0.0,0,Adjustable Race,None,1,None,None,AR-5381,None,750,1000,1998-06-01 00:00:00,0.0
1,None,0,0,0.0,0,Bearing Ball,None,2,None,None,BA-8327,None,750,1000,1998-06-01 00:00:00,0.0


In [33]:
#Populate employees dimension from MySQL Server - Dimension 2 

In [34]:
#sql_employees = "SELECT * FROM adventureworks.dim_employee_vw;"
#df_employees = get_sql_dataframe(sql_employees,**mysql_args2)
#df_employees.head(2)

In [85]:
sql_employees = "SELECT * FROM adventureworks.dim_employee_vw"
df_employees = get_mysql_dataframe(spark, sql_employees, **mysql_args)
df_employees.head(2)

[Row(EmployeeID=1, NationalIDNumber='14417807', LoginID='adventure-works\\guy1', ManagerID=16, FirstName='Guy', MiddleName='R', LastName='Gilbert', Title='Production Technician - WC60', EmailAddress='guy1@adventure-works.com', EmailPromotion=0, Phone='320-555-0195', BirthDate=datetime.datetime(1972, 5, 15, 0, 0), MaritalStatus='M', Gender='M', HireDate=datetime.datetime(1996, 7, 31, 0, 0), SalariedFlag=False, VacationHours=21, SickLeaveHours=30, CurrentFlag=True),
 Row(EmployeeID=2, NationalIDNumber='253022876', LoginID='adventure-works\\kevin0', ManagerID=6, FirstName='Kevin', MiddleName='F', LastName='Brown', Title='Marketing Assistant', EmailAddress='kevin0@adventure-works.com', EmailPromotion=2, Phone='150-555-0189', BirthDate=datetime.datetime(1977, 6, 3, 0, 0), MaritalStatus='S', Gender='M', HireDate=datetime.datetime(1997, 2, 26, 0, 0), SalariedFlag=False, VacationHours=42, SickLeaveHours=41, CurrentFlag=True)]

In [86]:
drop_cols = ['EmailPromotion','SalariedFlag','CurrentFlag']
df_employees.drop(*drop_cols)

DataFrame[EmployeeID: int, NationalIDNumber: string, LoginID: string, ManagerID: int, FirstName: string, MiddleName: string, LastName: string, Title: string, EmailAddress: string, Phone: string, BirthDate: timestamp, MaritalStatus: string, Gender: string, HireDate: timestamp, VacationHours: int, SickLeaveHours: int]

In [87]:
df_employees.write.saveAsTable(f"{dest_database}.employees", mode="overwrite")

In [88]:
spark.sql(f"DESCRIBE EXTENDED {dest_database}.employees;").show()
spark.sql(f"SELECT * FROM {dest_database}.employees LIMIT 2").toPandas()

+----------------+------------+-------+
|        col_name|   data_type|comment|
+----------------+------------+-------+
|      EmployeeID|         int|   NULL|
|NationalIDNumber| varchar(15)|   NULL|
|         LoginID|varchar(256)|   NULL|
|       ManagerID|         int|   NULL|
|       FirstName| varchar(50)|   NULL|
|      MiddleName| varchar(50)|   NULL|
|        LastName| varchar(50)|   NULL|
|           Title| varchar(50)|   NULL|
|    EmailAddress| varchar(50)|   NULL|
|  EmailPromotion|         int|   NULL|
|           Phone| varchar(25)|   NULL|
|       BirthDate|   timestamp|   NULL|
|   MaritalStatus|  varchar(1)|   NULL|
|          Gender|  varchar(1)|   NULL|
|        HireDate|   timestamp|   NULL|
|    SalariedFlag|     boolean|   NULL|
|   VacationHours|         int|   NULL|
|  SickLeaveHours|         int|   NULL|
|     CurrentFlag|     boolean|   NULL|
|                |            |       |
+----------------+------------+-------+
only showing top 20 rows



,EmployeeID,NationalIDNumber,LoginID,ManagerID,FirstName,MiddleName,LastName,Title,EmailAddress,EmailPromotion,Phone,BirthDate,MaritalStatus,Gender,HireDate,SalariedFlag,VacationHours,SickLeaveHours,CurrentFlag
0,1,14417807,adventure-works\guy1,16,Guy,R,Gilbert,Production Technician - WC60,guy1@adventure-works.com,0,320-555-0195,1972-05-15,M,M,1996-07-31,False,21,30,True
1,2,253022876,adventure-works\kevin0,6,Kevin,F,Brown,Marketing Assistant,kevin0@adventure-works.com,2,150-555-0189,1977-06-03,S,M,1997-02-26,False,42,41,True


In [40]:
#Populate customers dimension from MySQl Server - Dimension 3 

In [41]:
#sql_customers = "SELECT * FROM adventureworks.dim_customers_vw;"
#df_customers = get_sql_dataframe(sql_customers,**mysql_args2)
#df_customers.head(2)

In [42]:
sql_customers = "SELECT * FROM adventureworks.dim_customers_vw"
df_customers = get_mysql_dataframe(spark, sql_customers, **mysql_args)
df_customers.head(2)

[Row(CustomerID=1, AccountNumber='AW00000001', CustomerType='S', AddressType='Main Office', AddressLine1='2251 Elliot Avenue', AddressLine2=None, City='Seattle', StateProvinceCode='WA ', State_Province='Washington', IsOnlyStateProvinceFlag=False, PostalCode='98104', CountryRegionCode='US', Country_Region='United States', Sales Territory Group='North America', Sales Territory='Northwest'),
 Row(CustomerID=2, AccountNumber='AW00000002', CustomerType='S', AddressType='Shipping', AddressLine1='7943 Walnut Ave', AddressLine2=None, City='Renton', StateProvinceCode='WA ', State_Province='Washington', IsOnlyStateProvinceFlag=False, PostalCode='98055', CountryRegionCode='US', Country_Region='United States', Sales Territory Group='North America', Sales Territory='Northwest')]

In [43]:
drop_cols = ['AddressLine2', 'IsOnlyStateProvinceFlag']
df_customers.drop(*drop_cols)

DataFrame[CustomerID: int, AccountNumber: string, CustomerType: string, AddressType: string, AddressLine1: string, City: string, StateProvinceCode: string, State_Province: string, PostalCode: string, CountryRegionCode: string, Country_Region: string, Sales Territory Group: string, Sales Territory: string]

In [44]:
df_customers.write.saveAsTable(f"{dest_database}.customers", mode="overwrite")

In [45]:
spark.sql(f"DESCRIBE EXTENDED {dest_database}.customers;").show()
spark.sql(f"SELECT * FROM {dest_database}.customers LIMIT 2").toPandas()

+--------------------+------------------+-------+
|            col_name|         data_type|comment|
+--------------------+------------------+-------+
|          CustomerID|               int|   NULL|
|       AccountNumber|       varchar(10)|   NULL|
|        CustomerType|        varchar(1)|   NULL|
|         AddressType|       varchar(50)|   NULL|
|        AddressLine1|       varchar(60)|   NULL|
|        AddressLine2|       varchar(60)|   NULL|
|                City|       varchar(30)|   NULL|
|   StateProvinceCode|        varchar(3)|   NULL|
|      State_Province|       varchar(50)|   NULL|
|IsOnlyStateProvin...|           boolean|   NULL|
|          PostalCode|       varchar(15)|   NULL|
|   CountryRegionCode|        varchar(3)|   NULL|
|      Country_Region|       varchar(50)|   NULL|
|Sales Territory G...|       varchar(50)|   NULL|
|     Sales Territory|       varchar(50)|   NULL|
|                    |                  |       |
|# Detailed Table ...|                  |       |


,CustomerID,AccountNumber,CustomerType,AddressType,AddressLine1,AddressLine2,City,StateProvinceCode,State_Province,IsOnlyStateProvinceFlag,PostalCode,CountryRegionCode,Country_Region,Sales Territory Group,Sales Territory
0,1,AW00000001,S,Main Office,2251 Elliot Avenue,None,Seattle,WA,Washington,False,98104,US,United States,North America,Northwest
1,2,AW00000002,S,Shipping,7943 Walnut Ave,None,Renton,WA,Washington,False,98055,US,United States,North America,Northwest


In [46]:
#sql_fact_sales_orders = "SELECT * FROM adventureworks.fact_sales_orders_vw;"
#df_fact_sales_orders = get_sql_dataframe(sql_fact_sales_orders,**mysql_args2)
#df_fact_sales_orders.head(2)

In [47]:
#Spark Version

In [48]:
sql_fact_sales_orders = "SELECT * FROM adventureworks.fact_sales_orders_vw"
df_fact_sales_orders = get_mysql_dataframe(spark, sql_fact_sales_orders, **mysql_args)
df_fact_sales_orders.head(2)

[Row(SalesOrderID=43659, RevisionNumber=1, OrderDate=datetime.datetime(2001, 7, 1, 0, 0), DueDate=datetime.datetime(2001, 7, 13, 0, 0), ShipDate=datetime.datetime(2001, 7, 8, 0, 0), Status=5, OnlineOrderFlag=False, SalesOrderNumber='SO43659', PurchaseOrderNumber='PO522145787', AccountNumber='10-4020-000676', CustomerID=676, ContactID=378, SalesPersonID=279, Sales Territory Group='North America', Sales Territory='Southeast', BillToAddressID=985, ShipToAddressID=985, ShipMethod='CARGO TRANSPORT 5', ShipBase=8.99, ShipRate=1.49, Credit Card Type='ColonialVoice', Credit Card Number='77777462752259', Credit Card ExpMonth=2, Credit Card ExpYear=2007, CreditCardApprovalCode='105041Vi84182', SubTotal=24643.9362, TaxAmt=1971.5149, Freight=616.0984, TotalDue=27231.5495, CarrierTrackingNumber='4911-403C-98', OrderQty=4, ProductID=711, UnitPrice=20.1865, LineTotal=80.746),
 Row(SalesOrderID=43659, RevisionNumber=1, OrderDate=datetime.datetime(2001, 7, 1, 0, 0), DueDate=datetime.datetime(2001, 7, 1

In [49]:
drop_cols = ['OnlineOrderFlag']
df_fact_sales_orders.drop(*drop_cols)

DataFrame[SalesOrderID: int, RevisionNumber: tinyint, OrderDate: timestamp, DueDate: timestamp, ShipDate: timestamp, Status: tinyint, SalesOrderNumber: string, PurchaseOrderNumber: string, AccountNumber: string, CustomerID: int, ContactID: int, SalesPersonID: int, Sales Territory Group: string, Sales Territory: string, BillToAddressID: int, ShipToAddressID: int, ShipMethod: string, ShipBase: double, ShipRate: double, Credit Card Type: string, Credit Card Number: string, Credit Card ExpMonth: tinyint, Credit Card ExpYear: int, CreditCardApprovalCode: string, SubTotal: double, TaxAmt: double, Freight: double, TotalDue: double, CarrierTrackingNumber: string, OrderQty: int, ProductID: int, UnitPrice: double, LineTotal: double]

In [50]:
df_fact_sales_orders.write.saveAsTable(f"{dest_database}.fact_sales_orders", mode="overwrite")

In [51]:
spark.sql(f"DESCRIBE EXTENDED {dest_database}.fact_sales_orders;").show()
spark.sql(f"SELECT * FROM {dest_database}.fact_sales_orders LIMIT 2").toPandas()

+--------------------+-----------+-------+
|            col_name|  data_type|comment|
+--------------------+-----------+-------+
|        SalesOrderID|        int|   NULL|
|      RevisionNumber|    tinyint|   NULL|
|           OrderDate|  timestamp|   NULL|
|             DueDate|  timestamp|   NULL|
|            ShipDate|  timestamp|   NULL|
|              Status|    tinyint|   NULL|
|     OnlineOrderFlag|    boolean|   NULL|
|    SalesOrderNumber|varchar(25)|   NULL|
| PurchaseOrderNumber|varchar(25)|   NULL|
|       AccountNumber|varchar(15)|   NULL|
|          CustomerID|        int|   NULL|
|           ContactID|        int|   NULL|
|       SalesPersonID|        int|   NULL|
|Sales Territory G...|varchar(50)|   NULL|
|     Sales Territory|varchar(50)|   NULL|
|     BillToAddressID|        int|   NULL|
|     ShipToAddressID|        int|   NULL|
|          ShipMethod|varchar(50)|   NULL|
|            ShipBase|     double|   NULL|
|            ShipRate|     double|   NULL|
+----------

,SalesOrderID,RevisionNumber,OrderDate,DueDate,ShipDate,Status,OnlineOrderFlag,SalesOrderNumber,PurchaseOrderNumber,AccountNumber,...,CreditCardApprovalCode,SubTotal,TaxAmt,Freight,TotalDue,CarrierTrackingNumber,OrderQty,ProductID,UnitPrice,LineTotal
0,43659,1,2001-07-01,2001-07-13,2001-07-08,5,False,SO43659,PO522145787,10-4020-000676,...,105041Vi84182,24643.9362,1971.5149,616.0984,27231.5495,4911-403C-98,4,711,20.1865,80.746
1,43659,1,2001-07-01,2001-07-13,2001-07-08,5,False,SO43659,PO522145787,10-4020-000676,...,105041Vi84182,24643.9362,1971.5149,616.0984,27231.5495,4911-403C-98,2,712,5.1865,10.373


In [52]:
#Populate fact purchase orders table from cloud-based file system (CSV)

In [53]:
#csv_file = os.path.join(batch_dir, 'fact_purchase_orders_vw.csv')

#df_fact_purchase_orders = pd.read_csv(csv_file, header=0, index_col=0)
#df_fact_purchase_orders.head()

In [54]:
fact_purchase_orders_csv = os.path.join(batch_dir, 'fact_purchase_orders_vw.csv')
print(fact_purchase_orders_csv)

df_fact_purchase_orders = spark.read.format('csv').options(header='true', inferSchema='true').load(fact_purchase_orders_csv)
df_fact_purchase_orders.toPandas().head(2)

/Users/trinachowdhury/Documents/GitHub/DS-2002/03-NoSQL/data/batch/fact_purchase_orders_vw.csv


,PurchaseOrderID,RevisionNumber,Status,EmployeeID,VendorID,ProductID,OrderQty,UnitPrice,LineTotal,OrderDate,...,ShipRate,ShipDate,SubTotal,TaxAmt,Freight,TotalDue,DueDate,ReceivedQty,RejectedQty,StockedQty
0,1,0,4,244,83,1,4,50.2600,201.0400,2001-05-17,...,2.99,2001-05-26,201.0400,16.0832,5.0260,222.1492,2001-05-31,3.0,0.0,3.0
1,2,0,1,231,32,360,3,45.5805,136.7415,2001-05-17,...,1.49,2001-05-26,272.1015,21.7681,6.8025,300.6721,2001-05-31,3.0,0.0,3.0


In [55]:
df_fact_purchase_orders.write.saveAsTable(f"{dest_database}.fact_purchase_orders", mode="overwrite")

In [56]:
spark.sql(f"DESCRIBE EXTENDED {dest_database}.fact_purchase_orders;").show()
spark.sql(f"SELECT * FROM {dest_database}.fact_purchase_orders LIMIT 2").toPandas()

+---------------+---------+-------+
|       col_name|data_type|comment|
+---------------+---------+-------+
|PurchaseOrderID|      int|   NULL|
| RevisionNumber|      int|   NULL|
|         Status|      int|   NULL|
|     EmployeeID|      int|   NULL|
|       VendorID|      int|   NULL|
|      ProductID|      int|   NULL|
|       OrderQty|      int|   NULL|
|      UnitPrice|   double|   NULL|
|      LineTotal|   double|   NULL|
|      OrderDate|timestamp|   NULL|
|     ShipMethod|   string|   NULL|
|       ShipBase|   double|   NULL|
|       ShipRate|   double|   NULL|
|       ShipDate|timestamp|   NULL|
|       SubTotal|   double|   NULL|
|         TaxAmt|   double|   NULL|
|        Freight|   double|   NULL|
|       TotalDue|   double|   NULL|
|        DueDate|timestamp|   NULL|
|    ReceivedQty|   double|   NULL|
+---------------+---------+-------+
only showing top 20 rows



,PurchaseOrderID,RevisionNumber,Status,EmployeeID,VendorID,ProductID,OrderQty,UnitPrice,LineTotal,OrderDate,...,ShipRate,ShipDate,SubTotal,TaxAmt,Freight,TotalDue,DueDate,ReceivedQty,RejectedQty,StockedQty
0,1,0,4,244,83,1,4,50.2600,201.0400,2001-05-17,...,2.99,2001-05-26,201.0400,16.0832,5.0260,222.1492,2001-05-31,3.0,0.0,3.0
1,2,0,1,231,32,360,3,45.5805,136.7415,2001-05-17,...,1.49,2001-05-26,272.1015,21.7681,6.8025,300.6721,2001-05-31,3.0,0.0,3.0


In [57]:
#Ran dim_date query in MySQL and added to adventureworks_dw (the destination database)
#Then, created a sql_dim_date dataframe to fix the date formatting of dates in the fact_purchase_orders_vw 

In [58]:
#sql_dim_date = "SELECT date_key, full_date FROM adventureworks_dw.dim_date;"
#df_dim_date = get_sql_dataframe(sql_dim_date, **mysql_args2)
#df_dim_date.full_date = df_dim_date.full_date.astype('datetime64[ns]').dt.date
#df_dim_date.head(2)

In [60]:
sql_dim_date = f"SELECT * FROM {mysql_args['db_name']}.dim_date"
df_dim_date = get_mysql_dataframe(spark, sql_dim_date, **mysql_args)
df_dim_date.head(2)

[Row(date_key=20000101, full_date=datetime.date(2000, 1, 1), date_name='2000/01/01 ', date_name_us='01/01/2000 ', date_name_eu='01/01/2000 ', day_of_week=7, day_name_of_week='Saturday  ', day_of_month=1, day_of_year=1, weekday_weekend='Weekend   ', week_of_year=52, month_name='January   ', month_of_year=1, is_last_day_of_month='N', calendar_quarter=1, calendar_year=2000, calendar_year_month='2000-01   ', calendar_year_qtr='2000Q1    ', fiscal_month_of_year=7, fiscal_quarter=3, fiscal_year=2000, fiscal_year_month='2000-07   ', fiscal_year_qtr='2000Q3    '),
 Row(date_key=20000102, full_date=datetime.date(2000, 1, 2), date_name='2000/01/02 ', date_name_us='01/02/2000 ', date_name_eu='02/01/2000 ', day_of_week=1, day_name_of_week='Sunday    ', day_of_month=2, day_of_year=2, weekday_weekend='Weekend   ', week_of_year=52, month_name='January   ', month_of_year=1, is_last_day_of_month='N', calendar_quarter=1, calendar_year=2000, calendar_year_month='2000-01   ', calendar_year_qtr='2000Q1    

In [61]:
df_dim_date.write.saveAsTable(f"{dest_database}.dim_date", mode="overwrite")

In [62]:
spark.sql(f"DESCRIBE EXTENDED {dest_database}.dim_date;").show()
spark.sql(f"SELECT * FROM {dest_database}.dim_date LIMIT 2").toPandas()

+--------------------+---------+-------+
|            col_name|data_type|comment|
+--------------------+---------+-------+
|            date_key|      int|   NULL|
|           full_date|     date|   NULL|
|           date_name| char(11)|   NULL|
|        date_name_us| char(11)|   NULL|
|        date_name_eu| char(11)|   NULL|
|         day_of_week|  tinyint|   NULL|
|    day_name_of_week| char(10)|   NULL|
|        day_of_month|  tinyint|   NULL|
|         day_of_year|      int|   NULL|
|     weekday_weekend| char(10)|   NULL|
|        week_of_year|  tinyint|   NULL|
|          month_name| char(10)|   NULL|
|       month_of_year|  tinyint|   NULL|
|is_last_day_of_month|  char(1)|   NULL|
|    calendar_quarter|  tinyint|   NULL|
|       calendar_year|      int|   NULL|
| calendar_year_month| char(10)|   NULL|
|   calendar_year_qtr| char(10)|   NULL|
|fiscal_month_of_year|  tinyint|   NULL|
|      fiscal_quarter|  tinyint|   NULL|
+--------------------+---------+-------+
only showing top

,date_key,full_date,date_name,date_name_us,date_name_eu,day_of_week,day_name_of_week,day_of_month,day_of_year,weekday_weekend,...,is_last_day_of_month,calendar_quarter,calendar_year,calendar_year_month,calendar_year_qtr,fiscal_month_of_year,fiscal_quarter,fiscal_year,fiscal_year_month,fiscal_year_qtr
0,20000101,2000-01-01,2000/01/01,01/01/2000,01/01/2000,7,Saturday,1,1,Weekend,...,N,1,2000,2000-01,2000Q1,7,3,2000,2000-07,2000Q3
1,20000102,2000-01-02,2000/01/02,01/02/2000,02/01/2000,1,Sunday,2,2,Weekend,...,N,1,2000,2000-01,2000Q1,7,3,2000,2000-07,2000Q3


In [63]:
spark.sql(f"USE {dest_database};")
spark.sql("SHOW TABLES").toPandas()

,namespace,tableName,isTemporary
0,adventureworks_dlh,customers,False
1,adventureworks_dlh,dim_date,False
2,adventureworks_dlh,employees,False
3,adventureworks_dlh,fact_purchase_orders,False
4,adventureworks_dlh,fact_sales_orders,False
5,adventureworks_dlh,products,False


In [72]:
df_purchase = spark.table("adventureworks_dlh.fact_purchase_orders").toPandas()

total = len(df_purchase)
third = math.ceil(total / 3)

df_purchase.iloc[0:third].to_json(
    os.path.join(purchase_orders_stream_dir, 'adventureworks_purchase_orders_01.json'),
    orient='records'
)
df_purchase.iloc[third:third*2].to_json(
    os.path.join(purchase_orders_stream_dir, 'adventureworks_purchase_orders_02.json'),
    orient='records'
)
df_purchase.iloc[third*2:].to_json(
    os.path.join(purchase_orders_stream_dir, 'adventureworks_purchase_orders_03.json'),
    orient='records'
)

In [ ]:
#8.0. Use PySpark Structured Streaming to Process (Hot Path) Purchase Orders Fact Data
#8.1. Verify the location of the source data files on the file system¶

In [89]:
get_file_info(purchase_orders_stream_dir)

,name,size,modification_time
0,adventureworks_purchase_orders_01.json,141835,2026-05-07 03:07:27.014317989
1,adventureworks_purchase_orders_02.json,142227,2026-05-07 03:07:27.015883445
2,adventureworks_purchase_orders_03.json,141288,2026-05-07 03:07:27.016581058


In [90]:
#8.2. Create the Bronze Layer: Stage Purchase Orders Fact table Data
#8.2.1. Read "Raw" JSON file data into a Stream¶

In [91]:
df_purchase_orders_bronze = (
    spark.readStream \
    .option("schemaLocation", purchase_orders_output_bronze) \
    .option("maxFilesPerTrigger", 1) \
    .option("multiLine", "true") \
    .json(purchase_orders_stream_dir)
)

df_purchase_orders_bronze.isStreaming

True

In [92]:
#8.2.2. Write the Streaming Data to a Parquet file

In [97]:
purchase_orders_checkpoint_bronze = os.path.join(purchase_orders_output_bronze, '_checkpoint')

purchase_orders_bronze_query = (
    df_purchase_orders_bronze
    # Add Current Timestamp and Input Filename columns for Traceability
    .withColumn("receipt_time", current_timestamp())
    .withColumn("source_file", input_file_name())
    # TODO: writeStream to 'purchase_orders_output_bronze' in 'append' mode
    .writeStream \
    .format("parquet") \
    .outputMode("append") \
    .queryName("purchase_orders_bronze")
    .trigger(availableNow=True) \
    .option("checkpointLocation", purchase_orders_checkpoint_bronze) \
    .option("compression", "snappy") \
    .start(purchase_orders_output_bronze)
)

In [98]:
print(f"Query ID: {purchase_orders_bronze_query.id}")
print(f"Query Name: {purchase_orders_bronze_query.name}")
print(f"Query Status: {purchase_orders_bronze_query.status}")

Query ID: 478d1979-3fab-457c-9744-9c6ffece73bc
Query Name: purchase_orders_bronze
Query Status: {'message': 'Stopped', 'isDataAvailable': False, 'isTriggerActive': False}


In [101]:
purchase_orders_bronze_query.awaitTermination()

In [127]:
get_file_info(purchase_orders_output_bronze)

,name,size,modification_time
0,.part-00000-46875b33-a637-45f4-a70d-c57ed15d85...,164,2026-05-07 03:14:28.283089876
1,.part-00000-6df1b24b-644b-44e2-857f-d3c97db8a9...,164,2026-05-07 03:14:28.466423988
2,.part-00000-edf64e3c-3f6d-4e90-ad2d-94ea39c9e3...,160,2026-05-07 03:14:28.111779690
3,part-00000-46875b33-a637-45f4-a70d-c57ed15d85f...,19680,2026-05-07 03:14:28.283128738
4,part-00000-6df1b24b-644b-44e2-857f-d3c97db8a93...,19591,2026-05-07 03:14:28.466455221
5,part-00000-edf64e3c-3f6d-4e90-ad2d-94ea39c9e37...,19241,2026-05-07 03:14:28.111832857


In [102]:
from pyspark.sql.functions import col
from pyspark.sql.types import IntegerType, LongType, DoubleType, DateType

# Create date dimension aliases (like Northwind)
df_dim_order_date = df_dim_date.select(
    col("date_key").alias("order_date_key"), 
    col("full_date").alias("order_full_date")
)
df_dim_ship_date = df_dim_date.select(
    col("date_key").alias("ship_date_key"), 
    col("full_date").alias("ship_full_date")
)
df_dim_due_date = df_dim_date.select(
    col("date_key").alias("due_date_key"), 
    col("full_date").alias("due_full_date")
)

In [104]:
from pyspark.sql.functions import col, from_unixtime, current_timestamp, input_file_name
from pyspark.sql.types import IntegerType, LongType, DoubleType, DateType, TimestampType

df_purchase_orders_silver = spark.readStream.format("parquet").load(purchase_orders_output_bronze) \
    .join(df_employees, "EmployeeID") \
    .join(df_products, "ProductID") \
    .join(df_dim_order_date, df_dim_order_date.order_full_date.cast(DateType()) == from_unixtime(col("OrderDate")/1000).cast(DateType()), "inner") \
    .join(df_dim_ship_date, df_dim_ship_date.ship_full_date.cast(DateType()) == from_unixtime(col("ShipDate")/1000).cast(DateType()), "left_outer") \
    .join(df_dim_due_date, df_dim_due_date.due_full_date.cast(DateType()) == from_unixtime(col("DueDate")/1000).cast(DateType()), "left_outer") \
    .select(
        col("PurchaseOrderID").cast(LongType()),
        col("RevisionNumber").cast(IntegerType()),
        col("Status").cast(IntegerType()),
        col("EmployeeID").cast(IntegerType()),
        col("VendorID").cast(IntegerType()),
        col("ProductID").cast(IntegerType()),
        col("OrderQty").cast(IntegerType()),
        col("UnitPrice").cast(DoubleType()),
        col("LineTotal").cast(DoubleType()),
        col("ShipMethod"),
        col("ShipBase").cast(DoubleType()),
        col("ShipRate").cast(DoubleType()),
        col("SubTotal").cast(DoubleType()),
        col("TaxAmt").cast(DoubleType()),
        col("Freight").cast(DoubleType()),
        col("TotalDue").cast(DoubleType()),
        col("ReceivedQty").cast(DoubleType()),
        df_dim_order_date.order_date_key.cast(LongType()),
        df_dim_ship_date.ship_date_key.cast(LongType()),
        df_dim_due_date.due_date_key.cast(LongType())
    )

In [105]:
df_purchase_orders_silver.isStreaming

True

In [106]:
df_purchase_orders_silver.printSchema()

root
 |-- PurchaseOrderID: long (nullable = true)
 |-- RevisionNumber: integer (nullable = true)
 |-- Status: integer (nullable = true)
 |-- EmployeeID: integer (nullable = true)
 |-- VendorID: integer (nullable = true)
 |-- ProductID: integer (nullable = true)
 |-- OrderQty: integer (nullable = true)
 |-- UnitPrice: double (nullable = true)
 |-- LineTotal: double (nullable = true)
 |-- ShipMethod: string (nullable = true)
 |-- ShipBase: double (nullable = true)
 |-- ShipRate: double (nullable = true)
 |-- SubTotal: double (nullable = true)
 |-- TaxAmt: double (nullable = true)
 |-- Freight: double (nullable = true)
 |-- TotalDue: double (nullable = true)
 |-- ReceivedQty: double (nullable = true)
 |-- order_date_key: long (nullable = true)
 |-- ship_date_key: long (nullable = true)
 |-- due_date_key: long (nullable = true)



In [116]:
purchase_orders_checkpoint_silver = os.path.join(purchase_orders_output_silver, '_checkpoint')

purchase_orders_silver_query = (
    df_purchase_orders_silver.writeStream \
    .format("parquet") \
    .outputMode("append") \
    .queryName("orders_silver")
    .trigger(availableNow = True) \
    .option("checkpointLocation", purchase_orders_checkpoint_silver) \
    .option("compression", "snappy") \
    .start(purchase_orders_output_silver)
)

In [117]:
print(f"Query ID: {orders_silver_query.id}")
print(f"Query Name: {orders_silver_query.name}")
print(f"Query Status: {orders_silver_query.status}")

Query ID: 42125028-7271-4f89-9a17-7c8240687246
Query Name: orders_silver
Query Status: {'message': 'Stopped', 'isDataAvailable': False, 'isTriggerActive': False}


In [118]:
from pyspark.sql.functions import sum, count, asc, desc

df_purchase_orders_gold = spark.readStream.format("parquet").load(purchase_orders_output_silver) \
    .join(df_products, "ProductID") \
    .join(df_employees, "EmployeeID") \
    .join(df_dim_order_date, "order_date_key") \
    .groupBy("ProductCategory", "FirstName", "LastName") \
    .agg(
        count("PurchaseOrderID").alias("order_count"),
        sum("TotalDue").alias("total_spend"),
        sum("OrderQty").alias("total_qty_ordered")
    ) \
    .orderBy(desc("total_spend"))

In [119]:
df_purchase_orders_gold.printSchema()

root
 |-- ProductCategory: string (nullable = true)
 |-- FirstName: string (nullable = true)
 |-- LastName: string (nullable = true)
 |-- order_count: long (nullable = false)
 |-- total_spend: double (nullable = true)
 |-- total_qty_ordered: long (nullable = true)



In [122]:
purchase_orders_gold_query = (
    df_purchase_orders_gold.writeStream \
    .format("memory") \
    .outputMode("complete") \
    .queryName("purchase_orders_by_category")
    .start()
)

In [123]:
wait_until_stream_is_ready(purchase_orders_gold_query, 1)

The stream has processed 2 batchs


In [124]:
df_fact_orders_by_product_category = spark.sql("SELECT * FROM purchase_orders_by_category")
df_fact_orders_by_product_category.printSchema()

root
 |-- ProductCategory: string (nullable = true)
 |-- FirstName: string (nullable = true)
 |-- LastName: string (nullable = true)
 |-- order_count: long (nullable = false)
 |-- total_spend: double (nullable = true)
 |-- total_qty_ordered: long (nullable = true)



In [125]:
df_fact_orders_by_product_category_gold_final = df_fact_orders_by_product_category \
    .select(
        col("ProductCategory").alias("Product Category"),
        col("FirstName").alias("First Name"),
        col("LastName").alias("Last Name"),
        col("order_count").alias("Order Count"),
        col("total_spend").alias("Total Spend"),
        col("total_qty_ordered").alias("Total Qty Ordered")
    ) \
    .orderBy(desc("Total Spend"))

In [126]:
df_fact_orders_by_product_category_gold_final.write.saveAsTable(f"{dest_database}.purchase_orders_by_category", mode="overwrite")

spark.sql(f"SELECT * FROM {dest_database}.purchase_orders_by_category").toPandas()

,Product Category,First Name,Last Name,Order Count,Total Spend,Total Qty Ordered
0,None,Reinout,Hillmann,55,4.946751e+05,12712
1,Accessories,Annette,Hill,10,4.929421e+05,5500
2,Accessories,Reinout,Hillmann,10,4.401745e+05,5500
3,Accessories,Fukiko,Ogisu,8,3.371791e+05,4400
4,Accessories,Linda,Meisner,5,2.894719e+05,2750
5,Components,Fukiko,Ogisu,8,2.763297e+05,3910
6,None,Arvind,Rao,28,2.672293e+05,5668
7,Components,Mikael,Sandberg,22,2.319375e+06,12100
8,Components,Linda,Meisner,18,1.011627e+06,9900
9,Components,Frank,Pellow,15,8.837247e+05,8250


In [129]:
spark.sql("USE adventureworks_dlh")
spark.sql("SHOW TABLES").toPandas()

,namespace,tableName,isTemporary
0,adventureworks_dlh,customers,False
1,adventureworks_dlh,dim_date,False
2,adventureworks_dlh,employees,False
3,adventureworks_dlh,fact_purchase_orders,False
4,adventureworks_dlh,fact_sales_orders,False
5,adventureworks_dlh,products,False
6,adventureworks_dlh,purchase_orders_by_category,False
7,,purchase_orders_by_category,True


In [134]:
spark.sql(f"""
    SELECT 
        `Product Category`,
        COUNT(`Order Count`) as TotalOrders,
        SUM(`Total Spend`) as TotalSpend,
        SUM(`Total Qty Ordered`) as TotalQty,
        AVG(`Total Spend`) as AvgOrderValue
    FROM {dest_database}.purchase_orders_by_category
    GROUP BY `Product Category`
    ORDER BY TotalSpend DESC
""").toPandas()

,Product Category,TotalOrders,TotalSpend,TotalQty,AvgOrderValue
0,Components,12,8.021694e+06,70210,668474.500892
1,None,12,6.638019e+06,138748,553168.215583
2,Accessories,12,3.231005e+06,38500,269250.419033


In [136]:
spark.stop()